In [1]:
"""Parquet -> train.bin / val.bin -> random batches.

The tokenizer is used ONCE, offline. Training reads a flat uint16 array of
token ids via memmap and samples random windows out of it -- that random
offset IS the shuffling. Documents are concatenated end to end, separated by
the specials the template already inserts.
"""

import json
import os
import random
import sys

import numpy as np
from ua_text_utils import clean_ua_str

# Keep false in notebooks: true + fork (compile/Pool) can kill the kernel.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def tk_convert(doc_iter, out_dir, tok, val_frac=0.005, clean_fn=None,
               workers=None, batch=1000, seed=0):
    """Returns (n_train_tokens, n_val_tokens, n_docs).

    val is split by DOCUMENT, not by position: a positional split would make
    val pure-wikipedia if wiki happened to be written last.
    """
    from tqdm.auto import tqdm
    clean_fn = clean_fn or clean_ua_str                    # noqa: F821
    rng = random.Random(seed)
    os.makedirs(out_dir, exist_ok=True)

    paths = {s: os.path.join(out_dir, f"{s}.bin") for s in ("train", "val")}
    files = {s: open(p, "wb") for s, p in paths.items()}
    counts = {"train": 0, "val": 0}
    ndocs = 0

    def flush(docs):
        nonlocal ndocs
        if not docs:
            return
        for enc, d in zip(tok.encode_batch(docs), docs):    # specials via template
            split = "val" if rng.random() < val_frac else "train"
            arr = np.asarray(enc.ids, dtype=np.uint16)
            arr.tofile(files[split])
            counts[split] += arr.size
            ndocs += 1

    buf = []
    if workers == 1:
        it = (clean_fn(d) for d in doc_iter)
    else:
        from multiprocessing import Pool
        pool = Pool(workers)
        it = pool.imap(clean_fn, doc_iter, chunksize=64)

    try:
        for out in tqdm(it, unit="doc", file=sys.stdout):
            if not out:
                continue
            buf.append(out)
            if len(buf) >= batch:
                flush(buf)
                buf = []
        flush(buf)
    finally:
        if workers != 1:
            pool.terminate()
        for f in files.values():
            f.close()

    meta = {"vocab_size": tok.get_vocab_size(), "dtype": "uint16",
            "n_docs": ndocs, **{f"n_{k}": v for k, v in counts.items()}}
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    tot = counts["train"] + counts["val"]
    print(f"{ndocs:,} docs -> {tot/1e6:.1f}M tokens "
          f"(train {counts['train']/1e6:.1f}M, val {counts['val']/1e6:.2f}M)")
    return counts["train"], counts["val"], ndocs


def tk_verify(out_dir, tok, split="train"):
    """Catch doubled specials, uint16 wraparound, one-sided normalisation."""
    data = np.memmap(os.path.join(out_dir, f"{split}.bin"), dtype=np.uint16,
                     mode="r")
    meta = json.load(open(os.path.join(out_dir, "meta.json")))
    bos = tok.token_to_id("<|bos|>")
    eos = tok.token_to_id("<|eos|>")

    nb, ne = int((data == bos).sum()), int((data == eos).sum())
    print(f"{len(data):,} tokens | BOS {nb:,} | EOS {ne:,}")
    assert nb == ne, "BOS/EOS mismatch -- doubled or missing specials"
    assert data.max() < meta["vocab_size"], "id >= vocab_size: uint16 wrapped"

    end = int(np.flatnonzero(data == eos)[0])
    txt = tok.decode(data[:end + 1].tolist(), skip_special_tokens=True)
    print(f"first doc, {end+1} tokens:\n  {txt[:200]!r}")
    return data


def tk_get_batch(out_dir, split, B, T, device="cpu"):
    """Random windows. Re-opening the memmap each call is deliberate: holding
    one open across a long run leaks page-cache references."""
    import torch
    data = np.memmap(os.path.join(out_dir, f"{split}.bin"), dtype=np.uint16,
                     mode="r")
    ix = np.random.randint(0, len(data) - T - 1, size=B)
    x = np.stack([data[i:i + T] for i in ix]).astype(np.int64)
    y = np.stack([data[i + 1:i + 1 + T] for i in ix]).astype(np.int64)
    x, y = torch.from_numpy(x), torch.from_numpy(y)
    if device != "cpu":
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    return x, y

In [2]:
from ua_text_utils import tk_sample_sources
from tokenizers import Tokenizer

TOKENIZER = Tokenizer.from_file("ua_bpe_8k.json")

MATERIALIZE_DIR = "/home/nevidomy/data/tokenized/ua_small_2.6"
SOURCES = [
    ("/home/nevidomy/data/fineweb-2/data/ukr_Cyrl/train/*.parquet", 120_000_000),
    ("/home/nevidomy/data/ruvimx/UkrLM-wiki/wikipedia/train-*-of-00012.parquet", 30_000_000),
]

#tk_convert(tk_sample_sources(SOURCES), MATERIALIZE_DIR, TOKENIZER)
tk_verify(MATERIALIZE_DIR, TOKENIZER)
x, y = tk_get_batch(MATERIALIZE_DIR, "train", B=2, T=32, device="cuda")

print(x,y)

45,747,356 tokens | BOS 43,036 | EOS 43,036
first doc, 250 tokens:
  '- 27.01.2021\n- 581\nXiaomi Baseus 3.1A - крутий тандем для вашого авто\nЦей цікавий набір складається з автомобільного зарядного пристрою і розгалуджувача, який дозволяє використовувати 2 зарядних прист'
tensor([[ 611,  409, 4173, 5157,  325,  345, 1944,  723,  278, 1258,  279,  374,
          345, 3762,  608, 7114, 1002,  331,  471, 3762,  608, 7114,  357,   13,
         2944, 1355,  430, 1132,  278, 5619, 3401,  772],
        [ 770, 4187, 4152, 3880,  294,  324, 1592,  589,  440, 3910,  281, 7595,
           15, 4804,  308, 2338,  357,  317,  397,  364,  274, 4008,  313,  808,
          281, 6325, 7230,  540, 1078,   13, 2000,  397]], device='cuda:0') tensor([[ 409, 4173, 5157,  325,  345, 1944,  723,  278, 1258,  279,  374,  345,
         3762,  608, 7114, 1002,  331,  471, 3762,  608, 7114,  357,   13, 2944,
         1355,  430, 1132,  278, 5619, 3401,  772,  586],
        [4187, 4152, 3880,  294,  324, 1592,  5

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import trange
from trainui.client import Tracker, SequenceError

# Avoid fork+tokenizer deadlocks that kill the Jupyter kernel during compile.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.set_float32_matmul_precision("high")
# 32 compile workers OOMs / wedges WSL easily on a 3060.
torch._inductor.config.compile_threads = 4

DEVICE = "cuda"
VOCAB_SIZE = TOKENIZER.get_vocab_size()

#def encode(data):
#    return torch.tensor([VOCAB_MAP[x] for x in data])
#
#def decode(t):
#    return ''.join(VOCAB[x] for x in t)

#DATA = encode(DATA).to(DEVICE)
#DATA_TRAIN = DATA[:int(0.9 * DATA.size(0))]
#DATA_TEST = DATA[int(0.9 * DATA.size(0)):]

#@torch.no_grad()
#def get_batch(data, width, batch_size):
#    idx = torch.randint(data.size(0)-width,(batch_size,1), device=data.device)
#    idx = idx + torch.arange(width+1, device=data.device)
#    return data[idx[:,:-1]],data[idx[:,1:]]

class MultiheadAttention(nn.Module):
    def __init__(self, width, embeddings, headsize, dropout_p=0.0):
        super().__init__()
        assert embeddings % headsize == 0, "Embeddings should be multiple of headsize"
        self.headsize = headsize
        self.headcount = embeddings // headsize
        self.dropout_p = dropout_p
        self.qkv = nn.Linear(embeddings, 3*embeddings, bias=False)
        #self.drop = nn.Dropout(dropout_p)
        self.proj = nn.Linear(embeddings, embeddings)
        #self.register_buffer("causal_mask", torch.tril(torch.ones((width, width), dtype=bool)))
        ang = torch.arange(0, width).view(-1,1) * torch.tensor([1000.0**(-2*i/headsize) for i in range(headsize//2)])
        self.register_buffer("rope_cos", torch.cos(ang))
        self.register_buffer("rope_sin", torch.sin(ang))

    def rope(self, x):
        N = x.shape[-2]
        xx, yy = x.chunk(2,-1)
        return torch.concat(
            [xx*self.rope_cos[:N,:] - yy*self.rope_sin[:N,:],
            xx*self.rope_sin[:N,:] + yy*self.rope_cos[:N,:]],
            dim=-1)

    def forward(self, x):
        B,T,E = x.shape
        H,W = self.headcount, self.headsize
        q,k,v = self.qkv(x).view(B,T,3,H,W).permute(2,0,3,1,4) # B,H,T,W
        q = self.rope(q)
        k = self.rope(k)
        #a = (q @ k.transpose(-2,-1)) * W**-0.5 # B,H,T,T
        #a = torch.masked_fill(a, ~self.causal_mask[:T,:T], float("-inf"))
        #a = F.softmax(a, dim=-1)
        #a = self.drop(a)
        #a = (a @ v)
        a = F.scaled_dot_product_attention(q,k,v, is_causal=True, dropout_p=(self.dropout_p if self.training else 0.0))
        a = a.transpose(-3,-2).reshape(B,T,E)
        return self.proj(a)

class FeedForward(nn.Module):
    def __init__(self, embeddings, dropout_p=0.0):
        super().__init__()
        self.lin1 = nn.Linear(embeddings, embeddings*4)
        self.gelu = nn.GELU()
        self.lin2 = nn.Linear(embeddings*4, embeddings)

    def forward(self, x):
        x = self.lin1(x)
        x = self.gelu(x)
        x = self.lin2(x)
        return x

class AttentionBlock(nn.Module):
    def __init__(self, width, embeddings, headsize, dropout_p=0.0):
        super().__init__()
        self.mha = MultiheadAttention(
            width=width, 
            embeddings=embeddings, 
            headsize=headsize, 
            dropout_p=dropout_p)
        self.ff = FeedForward(
            embeddings=embeddings,
            dropout_p=dropout_p)
        self.layn1 = nn.LayerNorm(embeddings)
        self.layn2 = nn.LayerNorm(embeddings)
        self.drop = nn.Dropout(dropout_p)

    def forward(self, x):
        x = x + self.drop(self.mha(self.layn1(x)))
        x = x + self.drop(self.ff(self.layn2(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, vocab_size, width, embeddings, blocks, headsize, dropout_p=0.0):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_width = width
        self.embed = nn.Embedding(vocab_size, embeddings)
        #self.embed_pos = nn.Embedding(width, embeddings)
        self.drop = nn.Dropout(dropout_p)
        self.blocks = nn.Sequential(*[
            AttentionBlock(
                    width=width,
                    embeddings=embeddings,
                    headsize=headsize,
                    dropout_p=dropout_p) 
                for _ in range(blocks)
        ])
        self.layn = nn.LayerNorm(embeddings)
        self.lmh = nn.Linear(embeddings, vocab_size, bias=False)
        self.embed.weight = self.lmh.weight
        #self.register_buffer("position_offsets", torch.arange(width))

    def forward(self, x, y=None):
        B,T = x.shape
        x = self.embed(x)# + self.embed_pos(self.position_offsets[:T])
        x = self.drop(x)
        x = self.blocks(x)
        x = self.layn(x)
        logits = self.lmh(x)
        if y is None:
            return logits
        else:
            return F.cross_entropy(logits.view(-1,self.vocab_size), y.view(-1))

    @torch.no_grad()
    def generate(self, limit, prompt=""):
        was_training = self.training
        self.eval()
        device = self.lmh.weight.device
        if prompt:
            out = list(TOKENIZER.encode(prompt).ids)
        else:
            out = [0]
        for _ in range(limit):
            x = torch.tensor(out[-self.context_width:], device=device).view(1, -1)
            logits = self(x)[0, -1, :].float()
            out.append(int(torch.multinomial(F.softmax(logits, dim=-1), 1).item()))
        self.train(was_training)
        return out if prompt else out[1:]

@torch.no_grad()
def get_batch(split, width, batch_size):
    return tk_get_batch(MATERIALIZE_DIR, split, B=batch_size, T=width, device="cuda")

@torch.no_grad()
def estimate_loss(model, batch_size=128):
    was_train = model.training
    model.eval()
    l_test, l_train = [
        model(*get_batch(split, model.context_width, batch_size)) for split in ["train", "val"] 
    ]
    model.train(was_train)
    return  l_test, l_train


m = Transformer(
    vocab_size=VOCAB_SIZE,
    width=256,
    embeddings=128,
    blocks=8,
    headsize=32,
    dropout_p=0.0
).to(DEVICE)
torch.cuda.empty_cache()
# First compile can look like a hang; if the kernel dies here, set COMPILE=False.
COMPILE = True
m = torch.compile(m) if COMPILE else m

def train(model, optimizer, scheduler, run, iterations, batch_size):
    pbar = trange(0,iterations, desc="Learning")
    for i in pbar:
        optimizer.zero_grad()
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss = model(*get_batch("train", model.context_width, batch_size))
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        run.log(iteration=i, batches=batch_size, train_loss=loss.item(), lr =optimizer.param_groups[0]['lr'], grad_norm=grad_norm.item()) 
        if i%100==99:
            l_ts,l_tr = estimate_loss(model)
            pbar.set_postfix({
                "test": f"{l_ts.item():.4f}",
                "train": f"{l_tr.item():.4f}"})
            run.log(iteration=i, test_loss=l_ts.item())  

print(f"Model Size = {sum(p.numel() for p in m.parameters()):,}")

TRAINING_ITERATIONS = 4000
MAX_LEARNING_RATE = 1e-3
o = optim.AdamW(m.parameters(), lr = MAX_LEARNING_RATE)
s = optim.lr_scheduler.CosineAnnealingLR(o, eta_min=MAX_LEARNING_RATE*0.1, T_max=TRAINING_ITERATIONS)

print(f"Initial Loss: {estimate_loss(m)}")
print("[sample]:\n", TOKENIZER.decode(m.generate(100), skip_special_tokens=True))

tracker = Tracker(
    model_id="gpt-ua-small", 
    description="Warmup runs for GPT UA Model", 
    param_count=sum(p.numel() for p in m.parameters()),
    context_width=m.context_width,
    disabled=False)

with tracker.start_run() as run:
    train(m,o,s,run, iterations=TRAINING_ITERATIONS, batch_size=64)

print(f"Final Loss: {estimate_loss(m)}")
print("[sample]:\n", TOKENIZER.decode(m.generate(100), skip_special_tokens=True))



Model Size = 2,631,936
Initial Loss: (tensor(9.1906, device='cuda:0'), tensor(9.1818, device='cuda:0'))
[sample]:
 рпавсякаствен Цель програму коштів Тоділось(б ознайомбір соц брон-к річ∀ насправді справа(б.)
«І виявилилоосіб найчастішеok разі можливі долу мо нашійський відом інфра підозрюМар alкої філ фото са гри втраілля стані Івана вип нівачаМиільки припу груп ділянливий опри відсот ключ кожному діяль собі фонду фас площіЛюазвичайовин-г датобхід Держав� саме публіка БуБез сели справу сут пошкод двжиотою величез повинен перша-п буває потребу їхніх(;сив Такі військСередчні валю


Learning: 100%|██████████| 4000/4000 [02:36<00:00, 25.58it/s, test=4.1096, train=4.0547]


Final Loss: (tensor(4.1196, device='cuda:0'), tensor(4.0475, device='cuda:0'))
[sample]:
 Тез - з зовнішньої породи
Перший одяг почуває значні умови для сильного пилу, масла в дез-березну та повсякденну кіоди тенізу, направить обряду після Відносом Елфур-Гортмент протягом 25 хвилин.Два кварцотрила Безпеа
Меніргііка посилюється індивідуальним викликам, тому ви не можете вам


In [22]:
"""Loss as a function of position in the sequence.

Mean loss over a sequence is NOT comparable across context lengths: early
positions have almost no context and high loss, so averaging over T=64 weights
them far more heavily than averaging over T=256. A longer-context run will
report lower loss for that reason alone, before any real improvement.

Two things this answers:
  - how much of a T=64 vs T=256 gap is just position averaging
  - whether the model actually uses long context (does the curve keep falling
    past position 64, or flatten out?)
"""

import numpy as np
import torch
import torch.nn.functional as F

def tk_chars_per_token(out_dir, tok, split="val", limit=20_000_000,
                       chunk=1_000_000, save=True):
    path = os.path.join(out_dir, f"{split}.bin")
    data = np.memmap(path, dtype=np.uint16, mode="r")
    n = int(min(len(data), limit))
 
    chars = 0
    for i in range(0, n, chunk):
        ids = data[i:min(i + chunk, n)].tolist()
        chars += len(tok.decode(ids, skip_special_tokens=True))
 
    cpt = chars / n
    print(f"{split}: {n:,} tokens -> {chars:,} chars  =  {cpt:.4f} chars/token")
 
    if save:
        mp = os.path.join(out_dir, "meta.json")
        meta = json.load(open(mp)) if os.path.exists(mp) else {}
        meta[f"chars_per_token_{split}"] = cpt
        json.dump(meta, open(mp, "w"), indent=2)
    return cpt

def tk_baselines(out_dir, split="train", eval_split="val", vocab_size=None,
                 chars_per_token=None):
    meta_path = os.path.join(out_dir, "meta.json")
    if vocab_size is None and os.path.exists(meta_path):
        vocab_size = json.load(open(meta_path))["vocab_size"]
 
    tr = np.memmap(os.path.join(out_dir, f"{split}.bin"), dtype=np.uint16,
                   mode="r")
    va_path = os.path.join(out_dir, f"{eval_split}.bin")
    va = np.memmap(va_path, dtype=np.uint16, mode="r") if os.path.exists(va_path) else tr
 
    V = int(vocab_size or max(tr.max(), va.max()) + 1)
    rows = []
 
    rows.append(("uniform  ln(V)", float(np.log(V))))
 
    # ---- unigram, fit on train, evaluated on val -------------------------
    cnt = np.bincount(np.asarray(tr, dtype=np.int64), minlength=V).astype(np.float64)
    p = (cnt + 1.0) / (cnt.sum() + V)                     # Laplace, avoids inf
    nll = -np.log(p)[np.asarray(va, dtype=np.int64)]
    rows.append(("unigram", float(nll.mean())))
 
    # ---- bigram ----------------------------------------------------------
    if V * V <= 200_000_000:
        prev = np.asarray(tr[:-1], dtype=np.int64)
        nxt = np.asarray(tr[1:], dtype=np.int64)
        joint = np.bincount(prev * V + nxt, minlength=V * V).astype(np.float32)
        joint = joint.reshape(V, V)
        joint += 1.0                                      # Laplace
        cond = joint / joint.sum(axis=1, keepdims=True)
        vp = np.asarray(va[:-1], dtype=np.int64)
        vn = np.asarray(va[1:], dtype=np.int64)
        b = -np.log(cond[vp, vn].astype(np.float64))
        rows.append(("bigram", float(b.mean())))
        del joint, cond
    else:
        rows.append(("bigram", float("nan")))
 
    print(f"vocab {V:,} | train {len(tr):,} tok | eval {len(va):,} tok\n")
    hdr = f"{'floor':<16}{'nats/token':>12}"
    if chars_per_token:
        hdr += f"{'nats/char':>12}{'bits/char':>12}"
    print(hdr)
    for name, v in rows:
        line = f"{name:<16}{v:>12.4f}"
        if chars_per_token:
            npc = v / chars_per_token
            line += f"{npc:>12.4f}{npc/np.log(2):>12.4f}"
        print(line)
    return dict(rows)

def tk_bigram_floor(out_dir, V, chars_per_token=None,
                    alphas=(1.0, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001),
                    lambdas=(0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99)):
    tr = np.asarray(np.memmap(os.path.join(out_dir, "train.bin"),
                              dtype=np.uint16, mode="r"), dtype=np.int64)
    va = np.asarray(np.memmap(os.path.join(out_dir, "val.bin"),
                              dtype=np.uint16, mode="r"), dtype=np.int64)
 
    prev, nxt = tr[:-1], tr[1:]
    vp, vn = va[:-1], va[1:]
 
    # counts once; ~V*V int64 == 537MB at V=8192
    joint = np.bincount(prev * V + nxt, minlength=V * V)
    row = np.bincount(prev, minlength=V)
    uni = np.bincount(tr, minlength=V).astype(np.float64)
    p_uni = (uni + 1.0) / (uni.sum() + V)
 
    c = joint[vp * V + vn].astype(np.float64)      # val pair counts
    r = row[vp].astype(np.float64)                 # val context counts
    del joint
 
    seen = (c > 0).mean()
    zero_ctx = (r == 0).mean()
    print(f"train {len(tr):,} tok | val {len(va):,} tok | V {V:,}")
    print(f"  add-1 puts {V/(len(tr)/V + V)*100:.0f}% of each row's mass "
          f"in smoothing")
    print(f"  {seen*100:.1f}% of val bigrams were seen in train")
    print(f"  {zero_ctx*100:.2f}% of val contexts unseen\n")
 
    L2 = np.log(2)
 
    def show(name, nll):
        line = f"  {name:<22}{nll:>10.4f}"
        if chars_per_token:
            npc = nll / chars_per_token
            line += f"{npc:>11.4f}{npc/L2:>11.4f}"
        print(line)
        return nll
 
    hdr = f"  {'estimator':<22}{'nats/tok':>10}"
    if chars_per_token:
        hdr += f"{'nats/char':>11}{'bits/char':>11}"
    print(hdr)
 
    best = {}
    for a in alphas:
        nll = -np.log((c + a) / (r + a * V)).mean()
        best[f"add-{a:g}"] = show(f"add-{a:g}", nll)
 
    print()
    for lam in lambdas:
        with np.errstate(divide="ignore", invalid="ignore"):
            mle = np.where(r > 0, c / np.maximum(r, 1), 0.0)
        p = lam * mle + (1 - lam) * p_uni[vn]
        nll = -np.log(p).mean()
        best[f"interp-{lam:g}"] = show(f"interp lambda={lam:g}", nll)
 
    k = min(best, key=best.get)
    print(f"\nbest: {k} = {best[k]:.4f} nats/token", end="")
    if chars_per_token:
        print(f"  ({best[k]/chars_per_token/L2:.4f} bits/char)")
    else:
        print()
    return best

cpt_train = tk_chars_per_token(MATERIALIZE_DIR, TOKENIZER, split="train")
cpt_val   = tk_chars_per_token(MATERIALIZE_DIR, TOKENIZER, split="val")

print("Cpt train:", cpt_train)
print("Cpt val:", cpt_val)





train: 20,000,000 tokens -> 67,202,640 chars  =  3.3601 chars/token
val: 212,391 tokens -> 700,583 chars  =  3.2986 chars/token
Cpt train: 3.360132
Cpt val: 3.2985531401989725


In [23]:
floors = tk_baselines(MATERIALIZE_DIR, chars_per_token=3.360132)
print(floors)

tk_bigram_floor(MATERIALIZE_DIR, V=8192, chars_per_token=3.2986)

vocab 8,192 | train 45,747,356 tok | eval 212,391 tok

floor             nats/token   nats/char   bits/char
uniform  ln(V)        9.0109      2.6817      3.8689
unigram               7.5876      2.2581      3.2578
bigram                5.4939      1.6350      2.3588
{'uniform  ln(V)': 9.010913347279288, 'unigram': 7.587630575386056, 'bigram': 5.49391467718934}
train 45,747,356 tok | val 212,391 tok | V 8,192
  add-1 puts 59% of each row's mass in smoothing
  96.3% of val bigrams were seen in train
  0.00% of val contexts unseen

  estimator               nats/tok  nats/char  bits/char
  add-1                     5.4939     1.6655     2.4029
  add-0.3                   5.2232     1.5834     2.2844
  add-0.1                   5.1068     1.5482     2.2336
  add-0.03                  5.0761     1.5389     2.2201
  add-0.01                  5.0918     1.5436     2.2270
  add-0.003                 5.1274     1.5544     2.2426
  add-0.001                 5.1658     1.5660     2.2593

  interp

{'add-1': np.float64(5.493914677953682),
 'add-0.3': np.float64(5.223161627414921),
 'add-0.1': np.float64(5.106833009365495),
 'add-0.03': np.float64(5.0760750588269286),
 'add-0.01': np.float64(5.091824289408048),
 'add-0.003': np.float64(5.127437991561793),
 'add-0.001': np.float64(5.165771251609231),
 'interp-0.3': np.float64(5.742150398263327),
 'interp-0.5': np.float64(5.429719132006258),
 'interp-0.7': np.float64(5.224954238459798),
 'interp-0.8': np.float64(5.150444291051307),
 'interp-0.9': np.float64(5.097917989834253),
 'interp-0.95': np.float64(5.08853932067409),
 'interp-0.99': np.float64(5.122598678870041)}

In [37]:
print(TOKENIZER.decode(get_batch("val", 1024,1)[0][0].tolist()))

чірньою компанією Fujifilm. Клінічні випробування препарату відбувлись за участю 340 пацієнтів в Ухані і Шеньчжені.
“Він має високий ступінь безпеки і явно ефективний в лікуванні”, – зробив висновок представник китайського міністерства науки і техніки Чжан Сіньмінь.
Пацієнти, яким дали ліки в Шеньчжені, мали негативний результат аналіз на вірус в середньому через чотири дні після зараження, тоді як ті, хто не вживав ліки – в середньому через одинадцять.Більше того, рентген підтвердив поліпшення стану легень приблизно у 91% пацієнтів, яких лікували Favipiravir, порівняно з 62% поліпшення без препарату. Favipiravir потребуватиме схвалення уряду Японії для повномасштабного застосування на пацієнтах з Covid-19. Це може статися вже в травні.Була на весіллі. На весіллі… коханого.
У голові туманилось. Однак взяла себе в руки: й оком не повела, що через те страждає. «Гірко!» разом з усіма вигукувала. Щось там вітальне голосно промовляла. Молодята її чули й приязно відповідали. Це так, мимохіть